In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# IBM Circuit function

> **Note:** * Qiskit Functions — це експериментальна функція, доступна лише користувачам тарифних планів IBM Quantum&reg; Premium Plan, Flex Plan та On-Prem (через IBM Quantum Platform API). Вони мають статус попереднього релізу та можуть змінюватися.

## Огляд
IBM&reg; Circuit function приймає [абстрактні PUB'и](./primitive-input-output) як вхідні дані й повертає пом'якшені очікувані значення як результат. Ця circuit function включає автоматизований і налаштований конвеєр, що дозволяє дослідникам зосередитися на відкритті алгоритмів і застосунків.

## Опис
Після надсилання свого PUB твої абстрактні схеми та спостережувані автоматично транспілюються, виконуються на апаратному забезпеченні та постобробляються для повернення пом'якшених очікуваних значень. Для цього поєднуються такі інструменти:

- [Qiskit Transpiler Service](./qiskit-transpiler-service), включно з автоматичним вибором кроків транспіляції на основі ШІ та евристичних кроків для перетворення абстрактних схем у апаратно-оптимізовані ISA-схеми
- [Придушення та пом'якшення помилок, необхідні для обчислень у масштабі практичного застосування](./error-mitigation-and-suppression-techniques), включно з twirling вимірювань і Gate, динамічним роз'єднанням, Twirled Readout Error eXtinction (TREX), Zero-Noise Extrapolation (ZNE) та Probabilistic Error Amplification (PEA)
- [Qiskit Runtime Estimator](./get-started-with-primitives), для виконання ISA PUB'ів на апаратному забезпеченні та повернення пом'якшених очікуваних значень

![IBM Circuit function](../docs/images/guides/ibm-circuit-function/ibm-circuit-function.svg)

## Початок роботи
Автентифікуйся за допомогою свого [API-ключа](http://quantum.cloud.ibm.com/) і вибери Qiskit Function таким чином. (Цей фрагмент коду передбачає, що ти вже [зберіг свій акаунт](/guides/functions#install-qiskit-functions-catalog-client) у локальному середовищі.)

In [1]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

function = catalog.load("ibm/circuit-function")

## Приклад
Щоб почати, спробуй цей базовий приклад:

In [2]:
from qiskit.circuit.random import random_circuit
from qiskit_ibm_runtime import QiskitRuntimeService

# You can skip this step if you have a target backend, e.g.
# backend_name = "ibm_brisbane"
# You'll need to specify the credentials when initializing QiskitRuntimeService, if they were not previously saved.
service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

circuit = random_circuit(num_qubits=2, depth=2, seed=42)
observable = "Z" * circuit.num_qubits
pubs = [(circuit, observable)]

job = function.run(
    # Use `backend_name=backend_name` if you didn't initialize a backend object
    backend_name=backend.name,
    pubs=pubs,
)

Перевір [статус](/guides/functions#check-job-status) завдання Qiskit Function або отримай [результати](/guides/functions#retrieve-results) таким чином:

In [3]:
print(job.status())
result = job.result()

QUEUED


The results have the same format as an [Estimator result](/docs/guides/primitive-input-output#estimator-output):

In [4]:
print(f"The result of the submitted job had {len(result)} PUB\n")
print(
    f"The associated PubResult of this job has the following DataBins:\n {result[0].data}\n"
)
print(f"And this DataBin has attributes: {result[0].data.keys()}")
print(
    f"The expectation values measured from this PUB are: \n{result[0].data.evs}"
)

The result of the submitted job had 1 PUB

The associated PubResult of this job has the following DataBins:
 DataBin(evs=np.ndarray(<shape=(), dtype=float64>), stds=np.ndarray(<shape=(), dtype=float64>), ensemble_standard_error=np.ndarray(<shape=(), dtype=float64>))

And this DataBin has attributes: dict_keys(['evs', 'stds', 'ensemble_standard_error'])
The expectation values measured from this PUB are: 
1.02116704805492


Результати мають той самий формат, що й [результат Estimator](/guides/primitive-input-output#estimator-output):

In [5]:
options = {"mitigation_level": 2}

job = function.run(backend_name=backend.name, pubs=pubs, options=options)

#### All available options

In addition to `mitigation_level`, the IBM Circuit function also provides a select number of advanced options that allow you to fine-tune the cost/accuracy trade-off. The following table shows all the available options:

| Option | Sub-option | Sub-sub-option | Description | Choices | Default |
|-|-|-|-|-|-|
| default_precision |  |  | The default precision to use for any PUB or `run()`<br/>call that does not specify one.<br/>Each input PUB can specify its own precision. If the `run()` method is given a precision, then that value is used for all PUBs in the `run()` call that do not specify their own.  | float > 0 | 0.015625 |
| max_execution_time |  |  | Maximum execution time in seconds, which is based<br/>on QPU usage (not wall clock time). QPU usage is the<br/>amount of time that the QPU is dedicated to processing your job. If a job exceeds this time limit, it is forcibly canceled. | Integer number of seconds in the range [1, 10800] |  |
| mitigation_level |  |  | How much error suppression and mitigation to apply. Refer to the [Mitigation level](#mitigation-level) section for more information about the methods used at each level. | 1 / 2 / 3 | 1 |
| optimization_level |  |  | How much optimization to perform on the circuits. [Higher levels](/docs/guides/set-optimization) generate more optimized circuits, at the expense of longer transpilation time. | 1 / 2 / 3 | 2 |
| dynamical_decoupling | enable |  | Whether to enable dynamical decoupling. Refer to [Error suppression and mitigation techniques](/docs/guides/error-mitigation-and-suppression-techniques#dynamical-decoupling) for the explanation of the method.  | True/False | True |
|  | sequence_type |  | Which dynamical decoupling sequence to use.<br/>* `XX`: use the sequence `tau/2 - (+X) - tau - (+X) - tau/2`<br/>* `XpXm`: use the sequence `tau/2 - (+X) - tau - (-X) - tau/2`<br/>* `XY4`: use the sequence<br/>`tau/2 - (+X) - tau - (+Y) - tau (-X) - tau - (-Y) - tau/2` | 'XX'/'XpXm'/'XY4' | 'XX' |
| twirling | enable_gates |  | Whether to apply 2-qubit Clifford gate twirling. | True/False | False |
|  | enable_measure |  | Whether to enable twirling of measurements. | True/False | True |
| resilience | measure_mitigation |  | Whether to enable TREX measurement error mitigation method. Refer to [Error suppression and mitigation techniques](/docs/guides/error-mitigation-and-suppression-techniques#twirled-readout-error-extinction-trex) for the explanation of the method.  | True/False | True |
|  | zne_mitigation |  | Whether to turn on Zero Noise Extrapolation error mitigation method. Refer to [Error suppression and mitigation techniques](/docs/guides/error-mitigation-and-suppression-techniques#zero-noise-extrapolation-zne) for the explanation of the method.  | True/False | False |
|  | zne | amplifier | Which technique to use for amplifying noise. One of: <br/> - `gate_folding` (default) uses 2-qubit gate folding to amplify noise. If the noise factor requires amplifying only a subset of the gates, then these gates are chosen randomly.<br/><br/> - `gate_folding_front` uses 2-qubit gate folding to amplify noise. If the noise factor requires amplifying only a subset of the gates, then these gates are selected from the front of the topologically ordered DAG circuit.<br/><br/> - `gate_folding_back` uses 2-qubit gate folding to amplify noise. If the noise factor requires amplifying only a subset of the gates, then these gates are selected from the back of the topologically ordered DAG circuit.<br/><br/> - `pea` uses a technique called Probabilistic error amplification (PEA) to amplify noise. Refer to [Error suppression and mitigation techniques](/docs/guides/error-mitigation-and-suppression-techniques#probabilistic-error-amplification-pea) for the explanation of the method.  | gate_folding / gate_folding_front / gate_folding_back / pea | gate_folding |
|  |  | noise_factors | Noise factors to use for noise amplification. | list of floats; each float >= 1 | (1, 1.5, 2) for PEA, and (1, 3, 5) otherwise. |
|  |  | extrapolator | Noise factors to evaluate the fit extrapolation models at. This option does not affect execution or model fitting in any way; it only determines the points at which the `extrapolator` objects are evaluated to be returned in the data fields called `evs_extrapolated` and `stds_extrapolated`. | one or more of `exponential`,`linear`, `double_exponential`,`polynomial_degree_(1 <= k <= 7)` | (`exponential`, `linear`) |
|  | pec_mitigation |  | Whether to turn on Probabilistic Error Cancellation error mitigation method. Refer to [Error suppression and mitigation techniques](/docs/guides/error-mitigation-and-suppression-techniques#probabilistic-error-cancellation-pec) for the explanation of the method.  | True/False | False |
|  | pec | max_overhead | The maximum circuit sampling overhead allowed, or `None` for no maximum. | None/ integer >1 | 100 |

In the following example, setting the mitigation level to 1 initially turns off ZNE mitigation, but setting `zne_mitigation` to `True` overrides the relevant setup from `mitigation_level`.

In [6]:
options = {"mitigation_level": 1, "resilience": {"zne_mitigation": True}}

## Вхідні дані
Перегляньте таблицю нижче, де наведено всі вхідні параметри, які приймає IBM Circuit function. Подальший розділ [Параметри](#options) детальніше описує доступні `options`.

| Назва      | Тип                       | Опис                                                                                                                                                                                                                         | Обов'язково | За замовчуванням                                                                  | Приклад                                  |
|-----------|----------------------------|-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|----------|--------------------------------------------------------------------------|------------------------------------------|
| backend_name   | str                        | Назва backend'у для виконання запиту.                                                                                                                                                                                              | Так      | Н/Д                                                                      | `ibm_fez`                                |
| pubs      | Iterable[EstimatorPubLike] | Ітерований набір абстрактних PUB-подібних (primitive unified bloc) об'єктів, таких як кортежі `(circuit, observables)` або `(circuit, observables, parameter_values)`. Дивись [Огляд PUB'ів](/guides/primitive-input-output#overview-of-pubs) для отримання додаткової інформації. Схеми можуть бути абстрактними (не-ISA). | Так      | Н/Д                                                                      | (circuit, observables, parameter_values) |
| options   | dict                       | Вхідні параметри. Дивись розділ **Параметри** для отримання деталей.                                                                                                                                                                                | Ні       | Дивись розділ **Параметри** для деталей.                                                   | `{"optimization_level": 3}`                |
| instance  | str                        | Хмарна назва ресурсу (CRN) інстансу для використання у вказаному форматі.                                                                                                                                                                                        | Ні       | Вибирається випадково, якщо твій акаунт має доступ до кількох інстансів. | `CRN`                   |

### Параметри
#### Структура
Подібно до примітивів Qiskit Runtime, параметри для IBM Circuit function можна вказувати у вигляді вкладеного словника. Часто використовувані параметри, такі як ``optimization_level`` та ``mitigation_level``, знаходяться на першому рівні. Інші, більш просунуті параметри згруповано за різними категоріями, наприклад ``resilience``.

#### Значення за замовчуванням
Якщо ти не вказуєш значення для параметра, використовується значення за замовчуванням, встановлене сервісом.

#### Рівень пом'якшення
IBM Circuit function також підтримує `mitigation_level`. Рівень пом'якшення вказує, скільки придушення та пом'якшення помилок застосовувати до завдання. Вищі рівні генерують точніші результати коштом більшого часу обробки. Ступінь зменшення помилок залежить від застосованого методу. Рівень пом'якшення абстрагує детальний вибір методів пом'якшення та придушення помилок, дозволяючи користувачам оцінювати компроміс між вартістю та точністю, прийнятний для їхнього застосунку. У таблиці нижче наведено відповідні методи для кожного рівня.

> **Note:** Попри схожість назв, `mitigation_level` застосовує інші техніки, ніж ті, що використовуються `resilience_level` в Estimator.

Подібно до ``resilience_level`` в Qiskit Runtime Estimator, ``mitigation_level`` визначає базовий набір відібраних параметрів. Будь-які параметри, які ти вказуєш вручну на додаток до рівня пом'якшення, застосовуються поверх базового набору параметрів, визначеного рівнем пом'якшення. Тому теоретично можна встановити рівень пом'якшення рівним 1, але потім вимкнути пом'якшення вимірювань, хоча це не рекомендується.

| **Рівень пом'якшення** | **Техніка** |
|:-:|:-:|
| 1 [За замовчуванням] | Dynamical decoupling + measurement twirling + TREX  |
| 2 | Рівень 1 + gate twirling + ZNE через gate folding |
| 3 | Рівень 1 + gate twirling + ZNE через PEA |

Наступний приклад демонструє встановлення рівня пом'якшення:

In [7]:
print(f"The result of the submitted job had {len(result)} PUB")
print(
    f"The expectation values measured from this PUB are: \n{result[0].data.evs}"
)
print(f"And the associated metadata is: \n{result[0].metadata}")

The result of the submitted job had 1 PUB
The expectation values measured from this PUB are: 
1.02116704805492
And the associated metadata is: 
{'shots': 4096, 'target_precision': 0.015625, 'circuit_metadata': {}, 'resilience': {}, 'num_randomizations': 32}


#### Усі доступні параметри
На додаток до `mitigation_level`, IBM Circuit function також надає певну кількість розширених параметрів, що дозволяють тонко налаштувати компроміс між вартістю та точністю. У таблиці нижче наведено всі доступні параметри:

| Параметр | Підпараметр | Під-підпараметр | Опис | Значення | За замовчуванням |
|-|-|-|-|-|-|
| default_precision |  |  | Точність за замовчуванням для будь-якого PUB або виклику `run()`,<br/>який не вказує власну.<br/>Кожен вхідний PUB може вказати свою точність. Якщо методу `run()` передано точність, то це значення використовується для всіх PUB'ів у виклику `run()`, які не вказали свою.  | float > 0 | 0.015625 |
| max_execution_time |  |  | Максимальний час виконання в секундах, що базується<br/>на використанні QPU (не на реальному часі). Використання QPU — це<br/>час, протягом якого QPU присвячено обробці твого завдання. Якщо завдання перевищує цей ліміт, воно примусово скасовується. | Ціле число секунд у діапазоні [1, 10800] |  |
| mitigation_level |  |  | Скільки придушення та пом'якшення помилок застосовувати. Дивись розділ [Рівень пом'якшення](#mitigation-level) для отримання додаткової інформації про методи, що використовуються на кожному рівні. | 1 / 2 / 3 | 1 |
| optimization_level |  |  | Скільки оптимізації виконувати для схем. [Вищі рівні](/guides/set-optimization) генерують більш оптимізовані схеми коштом більшого часу транспіляції. | 1 / 2 / 3 | 2 |
| dynamical_decoupling | enable |  | Чи вмикати dynamical decoupling. Дивись [Техніки придушення та пом'якшення помилок](/guides/error-mitigation-and-suppression-techniques#dynamical-decoupling) для пояснення методу.  | True/False | True |
|  | sequence_type |  | Яку послідовність dynamical decoupling використовувати.<br/>* `XX`: використовує послідовність `tau/2 - (+X) - tau - (+X) - tau/2`<br/>* `XpXm`: використовує послідовність `tau/2 - (+X) - tau - (-X) - tau/2`<br/>* `XY4`: використовує послідовність<br/>`tau/2 - (+X) - tau - (+Y) - tau (-X) - tau - (-Y) - tau/2` | 'XX'/'XpXm'/'XY4' | 'XX' |
| twirling | enable_gates |  | Чи застосовувати twirling двокубітних Clifford Gate. | True/False | False |
|  | enable_measure |  | Чи вмикати twirling вимірювань. | True/False | True |
| resilience | measure_mitigation |  | Чи вмикати метод пом'якшення помилок вимірювань TREX. Дивись [Техніки придушення та пом'якшення помилок](/guides/error-mitigation-and-suppression-techniques#twirled-readout-error-extinction-trex) для пояснення методу.  | True/False | True |
|  | zne_mitigation |  | Чи вмикати метод пом'якшення помилок Zero Noise Extrapolation. Дивись [Техніки придушення та пом'якшення помилок](/guides/error-mitigation-and-suppression-techniques#zero-noise-extrapolation-zne) для пояснення методу.  | True/False | False |
|  | zne | amplifier | Яку техніку використовувати для підсилення шуму. Одна з: <br/> - `gate_folding` (за замовчуванням) використовує folding двокубітних Gate для підсилення шуму. Якщо коефіцієнт шуму вимагає підсилення лише підмножини Gate, то ці Gate вибираються випадково.<br/><br/> - `gate_folding_front` використовує folding двокубітних Gate для підсилення шуму. Якщо коефіцієнт шуму вимагає підсилення лише підмножини Gate, то ці Gate вибираються з початку топологічно впорядкованого DAG-схеми.<br/><br/> - `gate_folding_back` використовує folding двокубітних Gate для підсилення шуму. Якщо коефіцієнт шуму вимагає підсилення лише підмножини Gate, то ці Gate вибираються з кінця топологічно впорядкованого DAG-схеми.<br/><br/> - `pea` використовує техніку Probabilistic error amplification (PEA) для підсилення шуму. Дивись [Техніки придушення та пом'якшення помилок](/guides/error-mitigation-and-suppression-techniques#probabilistic-error-amplification-pea) для пояснення методу.  | gate_folding / gate_folding_front / gate_folding_back / pea | gate_folding |
|  |  | noise_factors | Коефіцієнти шуму для підсилення шуму. | список чисел float; кожне float >= 1 | (1, 1.5, 2) для PEA, та (1, 3, 5) в інших випадках. |
|  |  | extrapolator | Коефіцієнти шуму для оцінки моделей екстраполяції. Цей параметр жодним чином не впливає на виконання або підгонку моделі; він лише визначає точки, у яких об'єкти `extrapolator` оцінюються та повертаються у полях даних `evs_extrapolated` та `stds_extrapolated`. | одне або кілька з `exponential`,`linear`, `double_exponential`,`polynomial_degree_(1 <= k <= 7)` | (`exponential`, `linear`) |
|  | pec_mitigation |  | Чи вмикати метод пом'якшення помилок Probabilistic Error Cancellation. Дивись [Техніки придушення та пом'якшення помилок](/guides/error-mitigation-and-suppression-techniques#probabilistic-error-cancellation-pec) для пояснення методу.  | True/False | False |
|  | pec | max_overhead | Максимально допустимі накладні витрати на вибірку схем, або `None` без максимуму. | None/ ціле число >1 | 100 |

У наступному прикладі встановлення рівня пом'якшення рівним 1 спочатку вимикає ZNE-пом'якшення, але встановлення `zne_mitigation` в `True` перевизначає відповідні налаштування з `mitigation_level`.

In [8]:
job = function.run(
    backend_name="bad_backend_name", pubs=pubs, options=options
)

print(job.result())

QiskitServerlessException: "Traceback (most recent call last):\n  File \"/runner/runner.py\", line 10, in run\n    func = CircuitFunction(**arguments)\n           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File \"/runner/circuit_function/circuit_function.py\", line 87, in __init__\n    self._backend = self._service.backend(\n                    ^^^^^^^^^^^^^^^^^^^^^^\n  File \"/usr/local/lib/python3.11/site-packages/qiskit_ibm_runtime/qiskit_runtime_service.py\", line 754, in backend\n    backends = self.backends(name, instance=instance, use_fractional_gates=use_fractional_gates)\n               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File \"/usr/local/lib/python3.11/site-packages/qiskit_ibm_runtime/qiskit_runtime_service.py\", line 497, in backends\n    raise QiskitBackendNotFoundError(\"No backend matches the criteria.\")\nqiskit.providers.exceptions.QiskitBackendNotFoundError: 'No backend matches the criteria.'\n"

## Вихідні дані
Результатом Circuit function є [PrimitiveResult](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PrimitiveResult), який містить два поля:

- Один або більше об'єктів [PubResult](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PubResult). Їх можна індексувати безпосередньо з `PrimitiveResult`.

- Метадані на рівні завдання.

Кожен `PubResult` містить поля `data` та `metadata`.

- Поле `data` містить щонайменше масив очікуваних значень (`PubResult.data.evs`) та масив стандартних похибок (`PubResult.data.stds`). Воно також може містити більше даних залежно від використаних параметрів.

- Поле `metadata` містить метадані на рівні PUB (`PubResult.metadata`).

Наступний фрагмент коду описує формат `PrimitiveResult` (та пов'язаного `PubResult`).